# SDPA Backend Diagnostic: `enable_gqa=True` vs Manual KV Expansion

Tests whether PyTorch's `enable_gqa=True` parameter now properly dispatches to the
flash attention backend on Ampere+ GPUs (A10G / L40), or whether our manual
`repeat_interleave` KV head expansion still provides better backend selection.

**Run this on a GPU machine** (sandbox L40 or production A10G).

## What we test

1. **Backend eligibility** — which SDPA backends accept GQA inputs with `enable_gqa=True`?
2. **Backend eligibility** — which backends accept expanded (equal-head) inputs?
3. **Memory comparison** — peak VRAM for each approach
4. **Timing comparison** — wall-clock latency
5. **Correctness** — both approaches produce the same output

In [ ]:
import torch
import torch.nn.functional as F

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device:          {torch.cuda.get_device_name(0)}")
    cap = torch.cuda.get_device_capability(0)
    print(f"Compute cap:     {cap[0]}.{cap[1]}")
    props = torch.cuda.get_device_properties(0)
    total_mem = getattr(props, "total_memory", None) or getattr(props, "total_mem", 0)
    print(f"VRAM:            {total_mem / 1024**3:.1f} GiB")
    print(f"\nFlash SDP available:           {torch.backends.cuda.flash_sdp_enabled()}")
    print(
        f"Memory-efficient SDP available: {torch.backends.cuda.mem_efficient_sdp_enabled()}"
    )
    print(f"Math SDP available:             {torch.backends.cuda.math_sdp_enabled()}")
else:
    raise RuntimeError("No CUDA device — run this on a GPU machine")

## Setup: InternVL3.5-8B GQA dimensions

InternVL3.5-8B uses **Qwen3** LLM backbone:
- **32 Q heads**, **8 KV heads** (GQA ratio 4:1)
- **head_dim = 128**
- Typical sequence lengths: 500--4500 tokens (depending on tile count)

In [ ]:
# InternVL3.5-8B / Qwen3 GQA configuration
BATCH = 1
NUM_Q_HEADS = 32
NUM_KV_HEADS = 8
HEAD_DIM = 128
DTYPE = torch.bfloat16
DEVICE = "cuda:0"

# Test at multiple sequence lengths (tiles -> tokens)
SEQ_LENGTHS = [512, 1024, 2048, 4096]

REPEAT_FACTOR = NUM_Q_HEADS // NUM_KV_HEADS
print(
    f"GQA config: {NUM_Q_HEADS} Q heads, {NUM_KV_HEADS} KV heads (repeat factor {REPEAT_FACTOR})"
)
print(f"dtype: {DTYPE}, device: {DEVICE}")

## Test 1: Backend Eligibility

Force each backend individually and check if it accepts GQA inputs with `enable_gqa=True`
vs manually expanded inputs.

In [ ]:
def test_backend(
    backend_name, enable_flash, enable_mem_eff, enable_math, q, k, v, use_gqa=False
):
    """Try running SDPA with a specific backend. Returns True if it succeeds."""
    try:
        with torch.backends.cuda.sdp_kernel(
            enable_flash=enable_flash,
            enable_mem_efficient=enable_mem_eff,
            enable_math=enable_math,
        ):
            kwargs = {}
            if use_gqa:
                kwargs["enable_gqa"] = True
            out = F.scaled_dot_product_attention(q, k, v, **kwargs)
            torch.cuda.synchronize()
            return True, out.shape
    except Exception as e:
        return False, str(e)[:120]


seq_len = 2048  # Representative mid-range length

# GQA inputs (mismatched heads)
q_gqa = torch.randn(BATCH, NUM_Q_HEADS, seq_len, HEAD_DIM, dtype=DTYPE, device=DEVICE)
k_gqa = torch.randn(BATCH, NUM_KV_HEADS, seq_len, HEAD_DIM, dtype=DTYPE, device=DEVICE)
v_gqa = torch.randn(BATCH, NUM_KV_HEADS, seq_len, HEAD_DIM, dtype=DTYPE, device=DEVICE)

# Expanded inputs (equal heads)
k_exp = k_gqa.repeat_interleave(REPEAT_FACTOR, dim=1)
v_exp = v_gqa.repeat_interleave(REPEAT_FACTOR, dim=1)

backends = [
    ("flash", True, False, False),
    ("mem_efficient", False, True, False),
    ("math", False, False, True),
]

print(f"Sequence length: {seq_len}")
print(
    f"Q shape: {q_gqa.shape}, KV shape: {k_gqa.shape}, Expanded KV shape: {k_exp.shape}"
)
print()

print(f"{'Backend':<16} {'enable_gqa=True':<20} {'Manual expansion':<20}")
print("-" * 56)
for name, en_flash, en_mem, en_math in backends:
    ok_gqa, info_gqa = test_backend(
        name, en_flash, en_mem, en_math, q_gqa, k_gqa, v_gqa, use_gqa=True
    )
    ok_exp, info_exp = test_backend(
        name, en_flash, en_mem, en_math, q_gqa, k_exp, v_exp, use_gqa=False
    )
    status_gqa = "YES" if ok_gqa else f"NO ({info_gqa})"
    status_exp = "YES" if ok_exp else f"NO ({info_exp})"
    print(f"{name:<16} {status_gqa:<20} {status_exp:<20}")

# Cleanup
del q_gqa, k_gqa, v_gqa, k_exp, v_exp
torch.cuda.empty_cache()

## Test 2: Which backend does the dispatcher *actually* choose?

With all backends enabled (default), PyTorch's dispatcher picks the fastest eligible one.
We use `torch.profiler` to see which kernel actually runs.

In [ ]:
from contextlib import contextmanager


@contextmanager
def profile_sdpa():
    """Profile a single SDPA call and report which kernel ran."""
    with torch.profiler.profile(
        activities=[torch.profiler.ProfilerActivity.CUDA],
        record_shapes=True,
    ) as prof:
        yield prof

    # Identify the attention kernel from profiler events
    events = prof.key_averages()
    kernel_names = []
    for evt in events:
        name = evt.key.lower()
        if any(
            kw in name
            for kw in ["flash", "efficient", "sdpa", "attention", "bmm", "baddbmm"]
        ):
            kernel_names.append((evt.key, evt.cuda_time_total))

    kernel_names.sort(key=lambda x: -x[1])
    return kernel_names


seq_len = 2048
q = torch.randn(BATCH, NUM_Q_HEADS, seq_len, HEAD_DIM, dtype=DTYPE, device=DEVICE)
k = torch.randn(BATCH, NUM_KV_HEADS, seq_len, HEAD_DIM, dtype=DTYPE, device=DEVICE)
v = torch.randn(BATCH, NUM_KV_HEADS, seq_len, HEAD_DIM, dtype=DTYPE, device=DEVICE)
k_exp = k.repeat_interleave(REPEAT_FACTOR, dim=1)
v_exp = v.repeat_interleave(REPEAT_FACTOR, dim=1)

# Warmup
for _ in range(3):
    F.scaled_dot_product_attention(q, k_exp, v_exp)
torch.cuda.synchronize()

print("=== enable_gqa=True (native GQA) ===")
with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CUDA],
    record_shapes=True,
) as prof_gqa:
    F.scaled_dot_product_attention(q, k, v, enable_gqa=True)
    torch.cuda.synchronize()

for evt in prof_gqa.key_averages():
    name = evt.key.lower()
    if any(
        kw in name
        for kw in ["flash", "efficient", "sdpa", "attention", "bmm", "baddbmm", "gemm"]
    ):
        print(f"  {evt.key}: {evt.cuda_time_total:.0f} us")

print()
print("=== Manual KV expansion ===")
with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CUDA],
    record_shapes=True,
) as prof_exp:
    F.scaled_dot_product_attention(q, k_exp, v_exp)
    torch.cuda.synchronize()

for evt in prof_exp.key_averages():
    name = evt.key.lower()
    if any(
        kw in name
        for kw in ["flash", "efficient", "sdpa", "attention", "bmm", "baddbmm", "gemm"]
    ):
        print(f"  {evt.key}: {evt.cuda_time_total:.0f} us")

del q, k, v, k_exp, v_exp
torch.cuda.empty_cache()

## Test 3: Memory Comparison

Measure peak GPU memory for each approach across sequence lengths.

In [ ]:
import gc


def measure_memory(seq_len, use_gqa):
    """Run SDPA and return peak memory delta in MiB."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    baseline = torch.cuda.memory_allocated()

    q = torch.randn(BATCH, NUM_Q_HEADS, seq_len, HEAD_DIM, dtype=DTYPE, device=DEVICE)
    k = torch.randn(BATCH, NUM_KV_HEADS, seq_len, HEAD_DIM, dtype=DTYPE, device=DEVICE)
    v = torch.randn(BATCH, NUM_KV_HEADS, seq_len, HEAD_DIM, dtype=DTYPE, device=DEVICE)

    if use_gqa:
        out = F.scaled_dot_product_attention(q, k, v, enable_gqa=True)
    else:
        k = k.repeat_interleave(REPEAT_FACTOR, dim=1)
        v = v.repeat_interleave(REPEAT_FACTOR, dim=1)
        out = F.scaled_dot_product_attention(q, k, v)

    torch.cuda.synchronize()
    peak = torch.cuda.max_memory_allocated()

    del q, k, v, out
    torch.cuda.empty_cache()

    return (peak - baseline) / (1024**2)  # MiB


print(
    f"{'Seq Length':<12} {'enable_gqa (MiB)':<20} {'Manual expand (MiB)':<20} {'Savings (MiB)':<15}"
)
print("-" * 67)

for seq_len in SEQ_LENGTHS:
    try:
        mem_gqa = measure_memory(seq_len, use_gqa=True)
    except Exception as e:
        mem_gqa = f"FAIL: {e!s:.40s}"

    try:
        mem_exp = measure_memory(seq_len, use_gqa=False)
    except Exception as e:
        mem_exp = f"FAIL: {e!s:.40s}"

    if isinstance(mem_gqa, float) and isinstance(mem_exp, float):
        savings = mem_exp - mem_gqa
        print(f"{seq_len:<12} {mem_gqa:<20.1f} {mem_exp:<20.1f} {savings:<15.1f}")
    else:
        print(f"{seq_len:<12} {mem_gqa!s:<20} {mem_exp!s:<20}")

## Test 4: Timing Comparison

Benchmark latency across sequence lengths with proper CUDA synchronization.

In [ ]:
import time

N_WARMUP = 10
N_ITERS = 50


def benchmark(seq_len, use_gqa):
    """Return mean latency in ms."""
    q = torch.randn(BATCH, NUM_Q_HEADS, seq_len, HEAD_DIM, dtype=DTYPE, device=DEVICE)
    k = torch.randn(BATCH, NUM_KV_HEADS, seq_len, HEAD_DIM, dtype=DTYPE, device=DEVICE)
    v = torch.randn(BATCH, NUM_KV_HEADS, seq_len, HEAD_DIM, dtype=DTYPE, device=DEVICE)

    if not use_gqa:
        k = k.repeat_interleave(REPEAT_FACTOR, dim=1)
        v = v.repeat_interleave(REPEAT_FACTOR, dim=1)

    kwargs = {"enable_gqa": True} if use_gqa else {}

    # Warmup
    for _ in range(N_WARMUP):
        F.scaled_dot_product_attention(q, k, v, **kwargs)
    torch.cuda.synchronize()

    # Timed runs
    start = time.perf_counter()
    for _ in range(N_ITERS):
        F.scaled_dot_product_attention(q, k, v, **kwargs)
    torch.cuda.synchronize()
    elapsed = (time.perf_counter() - start) / N_ITERS * 1000  # ms

    del q, k, v
    torch.cuda.empty_cache()
    return elapsed


print(
    f"{'Seq Length':<12} {'enable_gqa (ms)':<18} {'Manual expand (ms)':<20} {'Speedup':<10}"
)
print("-" * 60)

for seq_len in SEQ_LENGTHS:
    try:
        t_gqa = benchmark(seq_len, use_gqa=True)
    except Exception as e:
        t_gqa = None
        print(f"{seq_len:<12} FAIL: {e!s:.60s}")
        continue

    try:
        t_exp = benchmark(seq_len, use_gqa=False)
    except Exception as e:
        t_exp = None
        print(f"{seq_len:<12} {t_gqa:<18.3f} FAIL: {e!s:.40s}")
        continue

    speedup = t_exp / t_gqa if t_gqa > 0 else 0
    winner = (
        "<-- gqa wins"
        if speedup > 1.05
        else ("<-- expand wins" if speedup < 0.95 else "~equal")
    )
    print(f"{seq_len:<12} {t_gqa:<18.3f} {t_exp:<20.3f} {speedup:<10.2f} {winner}")

## Test 5: Correctness Verification

Verify both approaches produce numerically equivalent output.

In [ ]:
seq_len = 1024

q = torch.randn(BATCH, NUM_Q_HEADS, seq_len, HEAD_DIM, dtype=DTYPE, device=DEVICE)
k = torch.randn(BATCH, NUM_KV_HEADS, seq_len, HEAD_DIM, dtype=DTYPE, device=DEVICE)
v = torch.randn(BATCH, NUM_KV_HEADS, seq_len, HEAD_DIM, dtype=DTYPE, device=DEVICE)

# enable_gqa approach
out_gqa = F.scaled_dot_product_attention(q, k, v, enable_gqa=True)

# Manual expansion approach
k_exp = k.repeat_interleave(REPEAT_FACTOR, dim=1)
v_exp = v.repeat_interleave(REPEAT_FACTOR, dim=1)
out_exp = F.scaled_dot_product_attention(q, k_exp, v_exp)

torch.cuda.synchronize()

max_diff = (out_gqa - out_exp).abs().max().item()
mean_diff = (out_gqa - out_exp).abs().mean().item()
rel_diff = ((out_gqa - out_exp).abs() / (out_exp.abs() + 1e-8)).mean().item()

print(f"Output shape:        {out_gqa.shape}")
print(f"Max absolute diff:   {max_diff:.2e}")
print(f"Mean absolute diff:  {mean_diff:.2e}")
print(f"Mean relative diff:  {rel_diff:.2e}")

# bf16 has ~7e-3 precision, so differences up to ~1e-2 are expected
if max_diff < 0.05:
    print("\nVERDICT: Outputs match (within bf16 precision)")
else:
    print(f"\nWARNING: Max diff {max_diff:.4f} exceeds bf16 tolerance — investigate!")

del q, k, v, k_exp, v_exp, out_gqa, out_exp
torch.cuda.empty_cache()

## Summary & Recommendation

Based on the results above:

| Scenario | Action |
|----------|--------|
| `enable_gqa=True` dispatches to **flash** backend | Switch to `enable_gqa=True` — saves ~50 MiB/layer from avoided KV expansion, flash handles GQA natively |
| `enable_gqa=True` falls back to **math** backend | Keep manual expansion — the O(N) flash/mem_efficient backends are worth the ~50 MiB cost |
| Both use flash with similar perf | Prefer `enable_gqa=True` for cleaner code and lower memory |

**Check Test 1 and Test 2 results** to determine which case applies to your GPU.